# Fine-tune Qwen3-Reranker-4B — Kaggle T4, QLoRA, phiên 12 giờ

**Đây là notebook fine-tune reranker, file 07.** Chọn GPU T4, Internet ON, dùng
Save Version → Save & Run All trong một phiên mới. Model mặc định duy nhất là
`Qwen/Qwen3-Reranker-4B`. Một GPU được dùng; T4 thứ hai không tự cộng VRAM.

Theo hướng dẫn chính thức Qwen: `model_type=qwen3_reranker`,
`task_type=generative_reranker`, `loss_type=generative_reranker`. Loss là
**binary cross-entropy của logit yes − logit no**, label 1/0; không dùng InfoNCE
của embedding hoặc loss language modeling. Một group có 1 positive + 1 negative
và được SWIFT mở thành **2 query/document pairs**. Accumulation là 8 groups.

Điều chỉnh phần cứng so với ví dụ full train nhiều GPU của Qwen: **QLoRA NF4
4-bit**, double quantization, FP16, rank 8, SDPA, gradient checkpointing, batch
1 group và max_length 512. T4 không dùng BF16/FlashAttention 2. Chưa đo VRAM/thời
gian 4B trên Kaggle thật; nếu OOM, giảm MAX_LENGTH=256 rồi tạo run mới.

Tài liệu ví dụ dùng MTEB/scidocs-reranking; notebook này mặc định dùng QASPER để
khớp miền tài liệu nghiên cứu của project. Định dạng role, template và loss giữ đúng
SWIFT. Negative là silver, cần kiểm duyệt nếu muốn dữ liệu chất lượng cao hơn.

**Ngân sách từ đầu phiên, gồm setup/download/data:** dừng train ở giờ 10,
hard stop worker ở 10h30, dừng evaluation ở giờ 11, chừa giờ cuối lưu Output.
Lưu checkpoint mỗi 100 optimizer steps hoặc khoảng 15 phút. Không mở tunnel,
không chờ secret, không merge 8GB weights trong phiên train. Output giữ adapter
và base commit SHA, cùng checkpoint optimizer/RNG để resume.

Nguồn chính thức đã đối chiếu:
- https://github.com/QwenLM/Qwen3-Embedding/blob/main/docs/training/SWIFT.md
- https://huggingface.co/Qwen/Qwen3-Reranker-4B
- https://github.com/modelscope/ms-swift/tree/v3.12.3
- https://huggingface.co/docs/peft/developer_guides/quantization

In [ ]:
import os, sys, json, time, subprocess, venv, importlib.metadata
from pathlib import Path

BASE_MODEL = "Qwen/Qwen3-Reranker-4B"
RUN_NAME = "qwen3-reranker-4b-" + time.strftime("%Y%m%d-%H%M%S", time.gmtime())
MAX_LENGTH = 512  # total prompt + instruction + query + document + assistant prefix
LEARNING_RATE = 6e-6  # starting point from Qwen's example; tune only on validation
MAX_STEPS = 500  # total step cap, may stop earlier at the 10h deadline
PAIR_LIMIT = 6000  # number of groups; each group expands to two pairs
PREPARED_DATA = ""  # optional prepared corpus directory under /kaggle/input
RESUME_CHECKPOINT = ""  # previous Output / RUN / resume; recipe in its parent
SESSION_ALREADY_ELAPSED_HOURS = 0.0  # set actual age if the GPU session is older
INSTRUCTION = "Given a research question, retrieve relevant passages that answer the question"

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/kaggle/temp/qwen-hf"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/temp/qwen-datasets"
os.environ["MODELSCOPE_CACHE"] = "/kaggle/temp/qwen-ms"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["NPROC_PER_NODE"] = "1"

In [ ]:
QWEN_LIBRARY_SOURCE = 'import hashlib, json, os, shutil, subprocess, time, math, re, random\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n"""QASPER paragraphs and gold evidence, preserving official paper-level splits."""\n\nimport hashlib\nimport json\nimport unicodedata\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Any\n\n\ndef read_jsonl(path: Path) -> list[dict[str, Any]]:\n    return [\n        json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()\n    ]\n\n\ndef write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(\n        "".join(json.dumps(row, ensure_ascii=False) + "\\n" for row in rows), encoding="utf-8"\n    )\n\n\ndef file_hash(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef normalize(text: str) -> str:\n    return " ".join(unicodedata.normalize("NFC", text).split())\n\n\ndef records(value: Any) -> list[dict[str, Any]]:\n    """HF\'s columnar Sequence and official raw list have the same canonical representation."""\n    if isinstance(value, list):\n        return value\n    if not value:\n        return []\n    lengths = {len(v) for v in value.values()}\n    if len(lengths) != 1:\n        raise ValueError("Malformed columnar sequence")\n    return [{k: v[i] for k, v in value.items()} for i in range(next(iter(lengths)))]\n\n\ndef prepare_qasper(source: Path, output: Path) -> dict[str, Any]:\n    corpus: list[dict[str, Any]] = []\n    queries: dict[str, list[dict[str, Any]]] = {}\n    pairs: list[dict[str, Any]] = []\n    paper_splits: dict[str, str] = {}\n    stats: Counter[str] = Counter()\n    hashes: dict[str, str] = {}\n    for split in ("train", "validation", "test"):\n        path = source / f"{split}.json"\n        hashes[split] = file_hash(path)\n        raw = json.loads(path.read_text(encoding="utf-8"))\n        papers = raw.items() if isinstance(raw, dict) else ((r["id"], r) for r in raw)\n        queries[split] = []\n        for paper_id, paper in papers:\n            paper_id = str(paper_id)\n            if paper_id in paper_splits:\n                raise ValueError(f"Paper leakage or duplicate: {paper_id}")\n            paper_splits[paper_id] = split\n            stats[f"{split}_papers"] += 1\n            lookup: dict[str, list[str]] = {}\n            paragraphs = [("Abstract", paper.get("abstract", ""))]\n            for section in records(paper["full_text"]):\n                paragraphs.extend(\n                    (section.get("section_name", ""), p) for p in section["paragraphs"]\n                )\n            for index, (section_name, text) in enumerate(paragraphs):\n                if not text.strip():\n                    continue\n                cid = paper_id + ":" + str(index)\n                row = {\n                    "id": cid,\n                    "paper_id": paper_id,\n                    "text": text.strip(),\n                    "section": section_name,\n                    "split": split,\n                }\n                corpus.append(row)\n                lookup.setdefault(normalize(text), []).append(cid)\n            for qa in records(paper["qas"]):\n                answers = [a["answer"] for a in records(qa["answers"])]\n                answerable = [a for a in answers if not a.get("unanswerable", False)]\n                gold = list(dict.fromkeys(e for a in answerable for e in a.get("evidence", [])))\n                textual = [e for e in gold if not e.startswith("FLOAT SELECTED")]\n                ids = sorted({cid for e in textual for cid in lookup.get(normalize(e), [])})\n                unmatched = [e for e in textual if normalize(e) not in lookup]\n                # Partially mapped gold would inflate recall: exclude such questions, report why.\n                reason = (\n                    "unanswerable"\n                    if not answerable\n                    else "non_text_evidence"\n                    if len(textual) != len(gold)\n                    else "unmatched_gold"\n                    if unmatched\n                    else "no_gold"\n                    if not ids\n                    else ""\n                )\n                question = {\n                    "id": str(qa["question_id"]),\n                    "question": qa["question"],\n                    "paper_id": paper_id,\n                    "relevant_ids": ids,\n                    "excluded_reason": reason,\n                    "split": split,\n                }\n                queries[split].append(question)\n                stats[f"{split}_{reason or \'eligible\'}"] += 1\n                if split == "train" and not reason:\n                    pairs.extend(\n                        {\n                            "query_id": question["id"],\n                            "question": question["question"],\n                            "positive_id": cid,\n                            "paper_id": paper_id,\n                            "source": "qasper_gold",\n                            "reviewed": True,\n                        }\n                        for cid in ids\n                    )\n    # Exact duplicate paragraphs across papers can contaminate evaluation too.\n    train_texts = {normalize(c["text"]) for c in corpus if c["split"] == "train"}\n    heldout_texts = {normalize(c["text"]) for c in corpus if c["split"] != "train"}\n    training_text = {c["id"]: normalize(c["text"]) for c in corpus if c["split"] == "train"}\n    original_pairs = len(pairs)\n    pairs = [p for p in pairs if training_text[p["positive_id"]] not in heldout_texts]\n    stats["train_pairs_excluded_duplicate_text"] = original_pairs - len(pairs)\n    stats["train_pairs"] = len(pairs)\n    output.mkdir(parents=True, exist_ok=True)\n    write_jsonl(output / "corpus.jsonl", corpus)\n    write_jsonl(output / "train_pairs.jsonl", pairs)\n    for split, rows in queries.items():\n        write_jsonl(output / f"{split}_queries.jsonl", rows)\n    manifest = {\n        "format_version": 2,\n        "dataset": "allenai/qasper",\n        "language": "en",\n        "source_hashes": hashes,\n        "paper_split": paper_splits,\n        "statistics": dict(stats),\n        "cross_split_duplicate_texts": len(train_texts & heldout_texts),\n        "corpus_sha256": file_hash(output / "corpus.jsonl"),\n        "pairs_sha256": file_hash(output / "train_pairs.jsonl"),\n    }\n    (output / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    return manifest\n\n\ndef load_split(data: Path, split: str) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:\n    corpus = [r for r in read_jsonl(data / "corpus.jsonl") if r["split"] == split]\n    queries = [q for q in read_jsonl(data / f"{split}_queries.jsonl") if not q["excluded_reason"]]\n    ids = {r["id"] for r in corpus}\n    if len(ids) != len(corpus) or not queries or not corpus:\n        raise ValueError("Missing eligible queries/corpus or duplicate corpus IDs")\n    if any(not q["relevant_ids"] or not set(q["relevant_ids"]) <= ids for q in queries):\n        raise ValueError("Gold evidence outside its split")\n    return corpus, queries\n\n\ndef validated_pairs(data: Path, synthetic: Path | None = None) -> list[dict[str, Any]]:\n    corpus = {r["id"]: r for r in read_jsonl(data / "corpus.jsonl")}\n    heldout_texts = {normalize(c["text"]) for c in corpus.values() if c["split"] != "train"}\n    pairs = read_jsonl(data / "train_pairs.jsonl")\n    if synthetic:\n        pairs += read_jsonl(synthetic)\n    result, seen = [], set()\n    for pair in pairs:\n        if pair.get("reviewed") is not True:\n            raise ValueError("Synthetic data must be reviewed by a human before training")\n        positive = corpus.get(pair["positive_id"])\n        if not positive or positive["split"] != "train" or positive["paper_id"] != pair["paper_id"]:\n            raise ValueError("Training evidence leaks held-out papers or has invalid provenance")\n        if normalize(positive["text"]) in heldout_texts:\n            raise ValueError("Training evidence duplicates held-out text")\n        key = (pair["question"], pair["positive_id"])\n        if key not in seen:\n            seen.add(key)\n            result.append({**pair, "positive": positive["text"]})\n    if not result:\n        raise ValueError("No valid training pairs")\n    return result\n\ndef download_qasper(destination, revision=""):\n    from datasets import load_dataset\n    from huggingface_hub import HfApi, hf_hub_download\n\n    destination = Path(destination)\n    api = HfApi()\n    revision = revision or api.dataset_info("allenai/qasper", revision="refs/convert/parquet").sha\n    files = api.list_repo_files("allenai/qasper", repo_type="dataset", revision=revision)\n    paths = {\n        split: [\n            hf_hub_download("allenai/qasper", name, repo_type="dataset", revision=revision)\n            for name in files\n            if name.endswith(".parquet") and f"/{split}/" in name\n        ]\n        for split in ("train", "validation", "test")\n    }\n    if any(not names for names in paths.values()):\n        raise ValueError("Official QASPER parquet conversion is missing a split")\n    dataset = load_dataset("parquet", data_files=paths)\n    destination.mkdir(parents=True, exist_ok=True)\n    for split in paths:\n        rows = {row["id"]: row for row in dataset[split]}\n        (destination / f"{split}.json").write_text(\n            json.dumps(rows, ensure_ascii=False), encoding="utf-8"\n        )\n    metadata = {"dataset": "allenai/qasper", "revision": revision, "license": "CC-BY-4.0"}\n    write_json(destination / "download.json", metadata)\n    return metadata\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding="utf-8")\n\nclass NotebookBM25:\n    """Inverted-index BM25; does not re-tokenize the corpus for every query."""\n\n    def __init__(self, texts):\n        self.lengths, self.postings = [], defaultdict(list)\n        for i, text in enumerate(texts):\n            counts = Counter(re.findall(r"\\w+", text.casefold()))\n            self.lengths.append(sum(counts.values()))\n            for term, count in counts.items():\n                self.postings[term].append((i, count))\n        self.average = max(sum(self.lengths) / max(len(texts), 1), 1)\n\n    def rank(self, query, count):\n        scores = defaultdict(float)\n        for term in set(re.findall(r"\\w+", query.casefold())):\n            postings = self.postings.get(term, [])\n            idf = math.log(1 + (len(self.lengths) - len(postings) + 0.5) / (len(postings) + 0.5))\n            for i, tf in postings:\n                scores[i] += (\n                    idf * tf * 2.5 / (tf + 1.5 * (0.25 + 0.75 * self.lengths[i] / self.average))\n                )\n        return sorted(scores, key=lambda i: (-scores[i], i))[:count]\nQWEN_REQUIREMENTS = [\n    "ms-swift==3.12.3",\n    "transformers==4.57.1",\n    "peft==0.17.1",\n    "trl==0.23.1",\n    "datasets==3.6.0",\n    "accelerate==1.11.0",\n    "bitsandbytes==0.48.2",\n    "huggingface-hub==0.36.0",\n    "sentence-transformers==5.1.2",\n    "numpy==2.2.6",\n    "pandas==2.2.3",\n    "pyarrow==20.0.0",\n    "scikit-learn==1.6.1",\n]\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\n    os.replace(temporary, path)\n\ndef session_start(clock_path, elapsed_hours=0):\n    """Retain the oldest observed start across cell reruns and kernel restarts."""\n    now = time.time()\n    try:\n        import psutil\n\n        kernel_start = psutil.Process(os.getpid()).create_time()\n    except ImportError:\n        kernel_start = now\n    started = min(kernel_start, now - elapsed_hours * 3600)\n    path = Path(clock_path)\n    if path.exists():\n        started = min(started, json.loads(path.read_text())["started_epoch"])\n    atomic_json(path, {"started_epoch": started})\n    return started\n\ndef install_qwen_environment(python, requirements, constraints, deadline):\n    remaining = min(1800, deadline - time.time())\n    if remaining <= 0:\n        raise RuntimeError("No setup budget remaining.")\n    subprocess.run(\n        [\n            str(python),\n            "-m",\n            "pip",\n            "install",\n            "--quiet",\n            "--prefer-binary",\n            "--only-binary=numpy,pandas,pyarrow,scikit-learn",\n            "--upgrade-strategy",\n            "only-if-needed",\n            "-c",\n            str(constraints),\n            *requirements,\n        ],\n        check=True,\n        timeout=remaining,\n    )\n\ndef swift_arguments(config):\n    """Flags verified against the pinned SWIFT wheel, not a moving latest API."""\n    result = [\n        "--model",\n        config["base_path"],\n        "--model_type",\n        "qwen3_emb",\n        "--task_type",\n        "embedding",\n        "--train_type",\n        "lora",\n        "--tuner_backend",\n        "peft",\n        "--use_hf",\n        "true",\n        "--quant_method",\n        "bnb",\n        "--quant_bits",\n        "4",\n        "--bnb_4bit_compute_dtype",\n        "float16",\n        "--bnb_4bit_quant_type",\n        "nf4",\n        "--bnb_4bit_use_double_quant",\n        "true",\n        "--torch_dtype",\n        "float16",\n        "--fp16",\n        "true",\n        "--bf16",\n        "false",\n        "--attn_impl",\n        "sdpa",\n        "--target_modules",\n        "all-linear",\n        "--lora_rank",\n        "8",\n        "--lora_alpha",\n        "16",\n        "--lora_dropout",\n        "0.05",\n        "--gradient_checkpointing",\n        "true",\n        "--dataset",\n        config["train_data"],\n        "--val_dataset",\n        config["validation_data"],\n        "--split_dataset_ratio",\n        "0",\n        "--max_length",\n        str(config["max_length"]),\n        "--truncation_strategy",\n        "delete",\n        "--loss_type",\n        "infonce",\n        "--label_names",\n        "labels",\n        "--per_device_train_batch_size",\n        "1",\n        "--per_device_eval_batch_size",\n        "1",\n        "--gradient_accumulation_steps",\n        "8",\n        "--learning_rate",\n        str(config["learning_rate"]),\n        "--num_train_epochs",\n        "1",\n        "--max_steps",\n        str(config["max_steps"]),\n        "--warmup_ratio",\n        "0.05",\n        "--eval_strategy",\n        "steps",\n        "--eval_steps",\n        "100",\n        "--save_strategy",\n        "steps",\n        "--save_steps",\n        "100",\n        "--save_total_limit",\n        "2",\n        "--save_only_model",\n        "false",\n        "--load_best_model_at_end",\n        "false",\n        "--metric_for_best_model",\n        "eval_loss",\n        "--greater_is_better",\n        "false",\n        "--logging_steps",\n        "5",\n        "--report_to",\n        "none",\n        "--dataloader_num_workers",\n        "0",\n        "--dataset_num_proc",\n        "1",\n        "--dataloader_drop_last",\n        "true",\n        "--seed",\n        "42",\n        "--data_seed",\n        "42",\n        "--output_dir",\n        config["output_dir"],\n        "--add_version",\n        "false",\n    ]\n    if config.get("resume"):\n        result += ["--resume_from_checkpoint", config["resume"]]\n    return result\n\ndef deadline_callback(deadline, progress_path, checkpoint_seconds=900, clock=time.time):\n    from transformers import TrainerCallback\n\n    class DeadlineCallback(TrainerCallback):\n        def __init__(self):\n            self.last_save = clock()\n\n        @staticmethod\n        def signature(model):\n            import torch\n\n            digest, count = hashlib.sha256(), 0\n            for name, parameter in model.named_parameters():\n                if not parameter.requires_grad:\n                    continue\n                value = parameter.detach().float().cpu()\n                if not torch.isfinite(value).all():\n                    raise RuntimeError(\n                        "Non-finite trainable weights; checkpoint is not deployable."\n                    )\n                digest.update(name.encode())\n                digest.update(value.numpy().tobytes())\n                count += value.numel()\n            if not count:\n                raise RuntimeError("No trainable adapter parameters found.")\n            return digest.hexdigest()\n\n        def on_train_begin(self, args, state, control, model=None, **kwargs):\n            self.initial_signature = self.signature(model)\n\n        def on_step_end(self, args, state, control, **kwargs):\n            now = clock()\n            atomic_json(progress_path, {"optimizer_steps": state.global_step, "updated": now})\n            if now >= deadline:\n                control.should_training_stop = True\n                control.should_save = state.global_step > 0\n                control.should_evaluate = False\n            elif now - self.last_save >= checkpoint_seconds:\n                control.should_save = True\n            return control\n\n        def on_save(self, args, state, control, **kwargs):\n            self.last_save = clock()\n            signature = self.signature(kwargs["model"])\n            atomic_json(\n                Path(progress_path).parent / "training_integrity.json",\n                {\n                    "optimizer_steps": state.global_step,\n                    "adapter_updated": signature != self.initial_signature,\n                },\n            )\n\n    return DeadlineCallback()\n\ndef run_bounded(command, log_path, deadline, env=None):\n    """A hung setup/evaluation cannot consume the entire 12h session."""\n    if time.time() >= deadline:\n        return {"returncode": None, "timed_out": True}\n    with Path(log_path).open("w", encoding="utf-8") as log:\n        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=env)\n        last_notice = time.time()\n        try:\n            while process.poll() is None:\n                if time.time() - last_notice >= 60:\n                    remaining = max(0, deadline - time.time()) / 3600\n                    print(\n                        f"{Path(log_path).name}: running; budget remaining {remaining:.2f}h",\n                        flush=True,\n                    )\n                    last_notice = time.time()\n                if time.time() >= deadline:\n                    process.terminate()\n                    try:\n                        process.wait(timeout=30)\n                    except subprocess.TimeoutExpired:\n                        process.kill()\n                        process.wait(timeout=15)\n                    return {"returncode": process.returncode, "timed_out": True}\n                time.sleep(2)\n            return {"returncode": process.returncode, "timed_out": False}\n        finally:\n            if process.poll() is None:\n                process.kill()\n                process.wait(timeout=15)\n\ndef complete_checkpoints(output_dir):\n    """Ignore incomplete writes after a hard timeout. Full checkpoints retain resume state."""\n    result = []\n    for path in Path(output_dir).rglob("checkpoint-*"):\n        required = (\n            "adapter_config.json",\n            "adapter_model.safetensors",\n            "trainer_state.json",\n            "optimizer.pt",\n            "scheduler.pt",\n            "rng_state.pth",\n        )\n        if not path.is_dir() or not all((path / name).is_file() for name in required):\n            continue\n        try:\n            state = json.loads((path / "trainer_state.json").read_text(encoding="utf-8"))\n            if state["global_step"] > 0:\n                result.append((state["global_step"], path, state))\n        except (OSError, ValueError, KeyError):\n            continue\n    return sorted(result)\n"""Qwen3 yes/no reranking with the official prompt and a pinned optional LoRA adapter."""\n\nimport json\nfrom pathlib import Path\n\nSYSTEM = (\n    "Judge whether the Document meets the requirements "\n    "based on the Query and the Instruct provided. "\n    \'Note that the answer can only be "yes" or "no".\'\n)\nPREFIX = f"<|im_start|>system\\n{SYSTEM}<|im_end|>\\n<|im_start|>user\\n"\nSUFFIX = "<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n"\nDEFAULT_INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"\n\n\ndef pair_token_ids(tokenizer, query, document, instruction, max_length, *, truncate=False):\n    prefix = tokenizer.encode(PREFIX, add_special_tokens=False)\n    suffix = tokenizer.encode(SUFFIX, add_special_tokens=False)\n    body = tokenizer.encode(\n        f"<Instruct>: {instruction}\\n<Query>: {query}\\n<Document>: {document}",\n        add_special_tokens=False,\n    )\n    room = max_length - len(prefix) - len(suffix)\n    if room < 1:\n        raise ValueError("MAX_LENGTH is smaller than the reranker prompt overhead.")\n    if len(body) > room and not truncate:\n        raise ValueError("Query/document exceeds MAX_LENGTH; gold truncation is forbidden.")\n    return prefix + body[:room] + suffix\n\n\ndef read_reranker_manifest(path):\n    root = Path(path)\n    value = json.loads((root / "research_model.json").read_text(encoding="utf-8"))\n    if (\n        value.get("format") != "swift_qwen3_reranker_adapter_v1"\n        or value.get("task_type") != "generative_reranker"\n        or value.get("model_family") != "qwen3_reranker"\n        or value.get("base_model") != "Qwen/Qwen3-Reranker-4B"\n        or value.get("fine_tuned") is not True\n        or value.get("optimizer_steps", 0) < 1\n        or len(value.get("base_revision", "")) != 40\n        or value.get("positive_token") != "yes"\n        or value.get("negative_token") != "no"\n        or not (root / "adapter_config.json").is_file()\n        or not (root / "adapter_model.safetensors").is_file()\n    ):\n        raise ValueError("Invalid trained Qwen3 reranker artifact; base fallback is forbidden.")\n    return value\n\n\nclass QwenRerankerScorer:\n    def __init__(\n        self,\n        base_path,\n        adapter_path=None,\n        *,\n        instruction=DEFAULT_INSTRUCTION,\n        max_length=512,\n        device="cuda:0",\n        quantized=True,\n    ):\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n\n        self.instruction, self.max_length = instruction, max_length\n        self.tokenizer = AutoTokenizer.from_pretrained(\n            str(base_path), padding_side="left", trust_remote_code=False, local_files_only=True\n        )\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        self.yes = self.tokenizer.convert_tokens_to_ids("yes")\n        self.no = self.tokenizer.convert_tokens_to_ids("no")\n        if (\n            self.yes == self.no\n            or self.yes is None\n            or self.no is None\n            or self.tokenizer.encode("yes", add_special_tokens=False) != [self.yes]\n            or self.tokenizer.encode("no", add_special_tokens=False) != [self.no]\n        ):\n            raise ValueError("The tokenizer must represent yes and no as distinct single tokens.")\n        options = {\n            "trust_remote_code": False,\n            "local_files_only": True,\n            "attn_implementation": "sdpa",\n        }\n        if quantized:\n            if not device.startswith("cuda"):\n                raise ValueError("This NF4 recipe requires a CUDA GPU.")\n            options.update(\n                {\n                    "torch_dtype": torch.float16,\n                    "device_map": {"": device},\n                    "quantization_config": BitsAndBytesConfig(\n                        load_in_4bit=True,\n                        bnb_4bit_quant_type="nf4",\n                        bnb_4bit_use_double_quant=True,\n                        bnb_4bit_compute_dtype=torch.float16,\n                    ),\n                }\n            )\n        else:\n            options["torch_dtype"] = torch.float32 if device == "cpu" else torch.float16\n        model = AutoModelForCausalLM.from_pretrained(str(base_path), **options)\n        if not quantized:\n            model.to(device)\n        if adapter_path:\n            from peft import PeftModel\n\n            model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)\n        self.model = model.eval()\n\n    def score(self, pairs, batch_size=1, *, truncate=False):\n        import torch\n\n        if batch_size < 1:\n            raise ValueError("batch_size must be positive")\n        scores = []\n        with torch.inference_mode():\n            for start in range(0, len(pairs), batch_size):\n                ids = [\n                    pair_token_ids(\n                        self.tokenizer, q, d, self.instruction, self.max_length, truncate=truncate\n                    )\n                    for q, d in pairs[start : start + batch_size]\n                ]\n                batch = self.tokenizer.pad({"input_ids": ids}, padding=True, return_tensors="pt")\n                batch = batch.to(self.model.device)\n                # Left padding leaves the final position valid for every pair.\n                logits = (\n                    self.model(**batch, logits_to_keep=1, use_cache=False).logits[:, -1].float()\n                )\n                margin = logits[:, self.yes] - logits[:, self.no]\n                if not torch.isfinite(margin).all():\n                    raise RuntimeError("Non-finite reranker score.")\n                scores.extend(margin.cpu().tolist())\n        return scores\n\n\ndef load_reranker_adapter(path, device="cuda:0"):\n    from huggingface_hub import snapshot_download\n\n    manifest = read_reranker_manifest(path)\n    base = snapshot_download(manifest["base_model"], revision=manifest["base_revision"])\n    return QwenRerankerScorer(\n        base,\n        path,\n        instruction=manifest["instruction"],\n        max_length=manifest["max_length"],\n        device=device,\n    )\n\n"""Pinned SWIFT generative-reranker/BCE recipe for Qwen3-Reranker-4B on Kaggle."""\n\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\n\nRECIPE = "qwen3-reranker-bce-v1"\n\n\ndef reranker_arguments(config):\n    options = dict(zip(swift_arguments(config)[::2], swift_arguments(config)[1::2], strict=True))\n    options.update(\n        {\n            "--model_type": "qwen3_reranker",\n            "--task_type": "generative_reranker",\n            "--loss_type": "generative_reranker",\n        }\n    )\n    return [item for pair in options.items() for item in pair]\n\n\ndef set_reranker_environment():\n    # One input group expands to exactly two query/document pairs on each device.\n    os.environ["MAX_POSITIVE_SAMPLES"] = "1"\n    os.environ["MAX_NEGATIVE_SAMPLES"] = "1"\n    os.environ["GENERATIVE_RERANKER_POSITIVE_TOKEN"] = "yes"\n    os.environ["GENERATIVE_RERANKER_NEGATIVE_TOKEN"] = "no"\n\n\ndef reranker_row(question, positive, negative, instruction):\n    return {\n        "messages": [\n            {"role": "system", "content": instruction},\n            {"role": "user", "content": question},\n        ],\n        "positive_messages": [[{"role": "assistant", "content": positive}]],\n        "negative_messages": [[{"role": "assistant", "content": negative}]],\n    }\n\n\ndef verify_swift_prompt(base_path, row, instruction, max_length):\n    """Fail early if the pinned SWIFT training template differs from inference token IDs."""\n    from swift.llm import get_model_tokenizer, get_template\n\n    _, processor = get_model_tokenizer(\n        str(base_path),\n        model_type="qwen3_reranker",\n        task_type="generative_reranker",\n        load_model=False,\n        use_hf=True,\n    )\n    template = get_template(\n        "qwen3_reranker", processor, max_length=max_length, truncation_strategy="delete"\n    )\n    template.set_mode("train")\n    encoded = template.encode(row)\n    question = row["messages"][-1]["content"]\n    expected = [\n        pair_token_ids(processor, question, row[key][0][0]["content"], instruction, max_length)\n        for key in ("positive_messages", "negative_messages")\n    ]\n    if encoded["input_ids"] != expected or encoded["labels"] != [1, 0]:\n        raise RuntimeError("SWIFT train/inference prompt mismatch; refusing to train.")\n    return {\n        "matched": True,\n        "labels": encoded["labels"],\n        "sequence_lengths": [len(ids) for ids in expected],\n    }\n\n\ndef prepare_reranker_data(data, tokenizer, instruction, max_length=512, limit=6000):\n    data = Path(data)\n    all_corpus = read_jsonl(data / "corpus.jsonl")\n    seen_ids, paper_splits = set(), {}\n    for item in all_corpus:\n        if (\n            item["id"] in seen_ids\n            or paper_splits.get(item["paper_id"], item["split"]) != item["split"]\n        ):\n            raise ValueError("Duplicate corpus ID or paper leakage between splits.")\n        seen_ids.add(item["id"])\n        paper_splits[item["paper_id"]] = item["split"]\n    heldout = {normalize(item["text"]) for item in all_corpus if item["split"] != "train"}\n    pairs = validated_pairs(data)\n    train_corpus, train_queries = load_split(data, "train")\n    val_corpus, val_queries = load_split(data, "validation")\n\n    def fits(question, document):\n        try:\n            pair_token_ids(tokenizer, question, document, instruction, max_length)\n            return True\n        except ValueError:\n            return False\n\n    def groups_for(corpus, queries, examples, count, split):\n        lookup = {item["id"]: item for item in corpus}\n        by_paper, gold = defaultdict(list), defaultdict(set)\n        for item in corpus:\n            by_paper[item["paper_id"]].append(item)\n        for query in queries:\n            gold[query["question"]].update(query["relevant_ids"])\n        indexes = {\n            paper: NotebookBM25([item["text"] for item in rows]) for paper, rows in by_paper.items()\n        }\n        result, provenance, excluded = [], [], 0\n        for item in examples:\n            question, cid = item["question"], item["positive_id"]\n            positive = lookup[cid]\n            forbidden = {normalize(lookup[i]["text"]) for i in gold[question]}\n            if not fits(question, positive["text"]):\n                excluded += 1\n                continue\n            rows = by_paper[item["paper_id"]]\n            candidates = indexes[item["paper_id"]].rank(question, len(rows))\n            candidates += [i for i in range(len(rows)) if i not in set(candidates)]\n            negative = next(\n                (\n                    rows[i]\n                    for i in candidates\n                    if normalize(rows[i]["text"]) not in forbidden\n                    and (split != "train" or normalize(rows[i]["text"]) not in heldout)\n                    and fits(question, rows[i]["text"])\n                ),\n                None,\n            )\n            if negative is None:\n                excluded += 1\n                continue\n            result.append(reranker_row(question, positive["text"], negative["text"], instruction))\n            provenance.append(\n                {\n                    "split": split,\n                    "query_id": item.get("query_id"),\n                    "paper_id": item["paper_id"],\n                    "positive_id": cid,\n                    "negative_id": negative["id"],\n                    "negative_label": "silver",\n                }\n            )\n            if len(result) >= count:\n                break\n        return result, provenance, excluded\n\n    random.Random(42).shuffle(pairs)\n    train, train_provenance, train_excluded = groups_for(\n        train_corpus, train_queries, pairs, limit, "train"\n    )\n    val_examples = [\n        {\n            "query_id": q["id"],\n            "question": q["question"],\n            "positive_id": q["relevant_ids"][0],\n            "paper_id": q["paper_id"],\n        }\n        for q in sorted(val_queries, key=lambda q: q["id"])\n    ]\n    validation, val_provenance, val_excluded = groups_for(\n        val_corpus, val_queries, val_examples, 64, "validation"\n    )\n    if len(train) < 16 or len(validation) < 4:\n        raise ValueError("Too few complete query/document pairs within MAX_LENGTH.")\n\n    index = NotebookBM25([item["text"] for item in val_corpus])\n    lookup = {item["id"]: item for item in val_corpus}\n    panel = []\n    for query in sorted(val_queries, key=lambda q: q["id"]):\n        if not all(fits(query["question"], lookup[cid]["text"]) for cid in query["relevant_ids"]):\n            continue\n        ranked = index.rank(query["question"], 100)\n        candidates = [\n            val_corpus[i] for i in ranked if fits(query["question"], val_corpus[i]["text"])\n        ][:20]\n        if not candidates:\n            continue\n        # Never insert gold: missed evidence remains a retrieval failure in the metrics.\n        panel.append(\n            {\n                "id": query["id"],\n                "question": query["question"],\n                "relevant_ids": query["relevant_ids"],\n                "candidates": candidates,\n            }\n        )\n        if len(panel) >= 16:\n            break\n    if not panel:\n        raise ValueError("No eligible validation queries for reranking evaluation.")\n    return (\n        train,\n        validation,\n        panel,\n        {\n            "train_groups": len(train),\n            "train_pairs": 2 * len(train),\n            "validation_groups": len(validation),\n            "validation_pairs": 2 * len(validation),\n            "excluded_train_groups": train_excluded,\n            "excluded_validation_groups": val_excluded,\n            "evaluation_queries": len(panel),\n            "negative_policy": "same-paper BM25 silver",\n            "provenance": train_provenance + val_provenance,\n        },\n    )\n\n\ndef prepare_reranker_worker(config):\n    import importlib.metadata\n\n    from huggingface_hub import HfApi, snapshot_download\n    from transformers import AutoTokenizer\n\n    run = Path(config["run"])\n    if config["base_model"] != "Qwen/Qwen3-Reranker-4B":\n        raise ValueError("This recipe trains Qwen3-Reranker-4B only.")\n    recipe = {}\n    if config.get("resume"):\n        recipe = json.loads((Path(config["resume"]).parent / "resume_recipe.json").read_text())\n        for key in ("recipe", "base_model", "instruction", "max_length", "learning_rate"):\n            if recipe.get(key) != config[key]:\n                raise ValueError(f"Resume recipe mismatch: {key}.")\n    config["base_revision"] = (\n        recipe.get("base_revision") or HfApi().model_info(config["base_model"]).sha\n    )\n    config["base_path"] = snapshot_download(config["base_model"], revision=config["base_revision"])\n    data = Path(config.get("prepared_data") or "/kaggle/temp/qwen-reranker-data")\n    if not config.get("prepared_data"):\n        raw = Path("/kaggle/temp/qwen-reranker-raw")\n        metadata = download_qasper(raw, recipe.get("dataset_revision", ""))\n        config["dataset_revision"] = metadata["revision"]\n        atomic_json(run / "dataset_download.json", metadata)\n        atomic_json(run / "dataset_manifest.json", prepare_qasper(raw, data))\n    tokenizer = AutoTokenizer.from_pretrained(config["base_path"], trust_remote_code=False)\n    train, validation, panel, stats = prepare_reranker_data(\n        data, tokenizer, config["instruction"], config["max_length"], config["pair_limit"]\n    )\n    for name, rows in (("train", train), ("validation", validation)):\n        path = run / f"{name}.jsonl"\n        write_jsonl(path, rows)\n        config[f"{name}_data"] = str(path)\n    config["data_sha256"] = {\n        key: hashlib.sha256(Path(config[key]).read_bytes()).hexdigest()\n        for key in ("train_data", "validation_data")\n    }\n    if recipe and recipe["data_sha256"] != config["data_sha256"]:\n        raise ValueError("Resume data differs; use the original dataset and pair limit.")\n    set_reranker_environment()\n    atomic_json(\n        run / "prompt_check.json",\n        verify_swift_prompt(\n            config["base_path"], train[0], config["instruction"], config["max_length"]\n        ),\n    )\n    config["packages"] = {\n        name: importlib.metadata.version(name)\n        for name in ("torch", "ms-swift", "transformers", "peft", "bitsandbytes", "datasets")\n    }\n    atomic_json(run / "config.json", config)\n    atomic_json(run / "data_statistics.json", stats)\n    atomic_json(run / "evaluation_panel.json", panel)\n\n\ndef train_reranker_worker(config):\n    set_reranker_environment()\n    from swift.llm import sft_main\n    from swift.plugin import extra_callbacks\n\n    if time.time() >= config["train_deadline"]:\n        raise RuntimeError("Training deadline expired before loading model.")\n    extra_callbacks.append(\n        deadline_callback(config["train_deadline"], Path(config["run"]) / "progress.json")\n    )\n    result = sft_main(reranker_arguments(config))\n    atomic_json(Path(config["run"]) / "train_result.json", result)\n\n\ndef export_reranker_adapter(config, status):\n    run = Path(config["run"])\n    integrity_path = run / "training_integrity.json"\n    integrity = json.loads(integrity_path.read_text()) if integrity_path.exists() else {}\n    if not integrity.get("adapter_updated"):\n        raise RuntimeError("No verified adapter weight update; no model is exported.")\n    checkpoints = [\n        item\n        for item in complete_checkpoints(config["output_dir"])\n        if item[0] <= integrity.get("optimizer_steps", 0)\n    ]\n    if not checkpoints:\n        raise RuntimeError("No complete resumable checkpoint; inspect train.log.")\n    _, latest, state = checkpoints[-1]\n    best = Path(state.get("best_model_checkpoint") or "").name\n    step, selected, _ = next(\n        (item for item in checkpoints if item[1].name == best), checkpoints[-1]\n    )\n    output = run / "qwen3-reranker-adapter"\n    output.mkdir(exist_ok=True)\n    for name in ("adapter_config.json", "adapter_model.safetensors"):\n        shutil.copy2(selected / name, output / name)\n    atomic_json(\n        output / "research_model.json",\n        {\n            "format": "swift_qwen3_reranker_adapter_v1",\n            "model_family": "qwen3_reranker",\n            "task_type": "generative_reranker",\n            "loss_type": "generative_reranker",\n            "loss": "BCEWithLogitsLoss(yes_logit - no_logit, binary_label)",\n            "positive_token": "yes",\n            "negative_token": "no",\n            "fine_tuned": True,\n            "optimizer_steps": step,\n            "base_model": config["base_model"],\n            "base_revision": config["base_revision"],\n            "instruction": config["instruction"],\n            "max_length": config["max_length"],\n            "recipe": RECIPE,\n            "selection": "best_eval_loss" if selected.name == best else "last_unvalidated",\n            "quantization": "bnb NF4 double quantization, FP16 compute",\n            "process_status": status,\n            "packages": config.get("packages", {}),\n            "data_sha256": config["data_sha256"],\n            "negative_policy": "same-paper BM25 silver, all known gold excluded",\n        },\n    )\n    if not (run / "resume").exists():\n        shutil.copytree(latest, run / "resume")\n    atomic_json(\n        run / "resume_recipe.json",\n        {\n            **{\n                key: config[key]\n                for key in (\n                    "base_model",\n                    "base_revision",\n                    "instruction",\n                    "max_length",\n                    "learning_rate",\n                    "data_sha256",\n                    "recipe",\n                )\n            },\n            "dataset_revision": config.get("dataset_revision", ""),\n        },\n    )\n    return output\n\n\ndef ranking_metrics(panel, scores=None):\n    ndcg, mrr, recall, candidate_recall = [], [], [], []\n    for i, item in enumerate(panel):\n        gold = set(item["relevant_ids"])\n        candidates = item["candidates"]\n        order = list(range(len(candidates)))\n        if scores is not None:\n            if len(scores[i]) != len(candidates) or not all(math.isfinite(x) for x in scores[i]):\n                raise ValueError("Missing or non-finite evaluation scores.")\n            order.sort(key=lambda j: (-scores[i][j], j))\n        relevance = [int(candidates[j]["id"] in gold) for j in order[:10]]\n        dcg = sum(hit / math.log2(rank + 2) for rank, hit in enumerate(relevance))\n        ideal = sum(1 / math.log2(rank + 2) for rank in range(min(10, len(gold))))\n        ndcg.append(dcg / ideal)\n        mrr.append(next((1 / (rank + 1) for rank, hit in enumerate(relevance) if hit), 0))\n        recall.append(sum(relevance) / len(gold))\n        candidate_recall.append(len({c["id"] for c in candidates} & gold) / len(gold))\n    if not panel:\n        raise ValueError("Cannot evaluate an empty panel.")\n    return {\n        "ndcg@10": sum(ndcg) / len(panel),\n        "mrr@10": sum(mrr) / len(panel),\n        "recall@10": sum(recall) / len(panel),\n        "candidate_recall@20": sum(candidate_recall) / len(panel),\n    }\n\n\ndef evaluate_reranker_worker(config):\n    run = Path(config["run"])\n    panel = json.loads((run / "evaluation_panel.json").read_text(encoding="utf-8"))\n    scorer = QwenRerankerScorer(\n        config["base_path"],\n        run / "qwen3-reranker-adapter",\n        instruction=config["instruction"],\n        max_length=config["max_length"],\n    )\n\n    def measure():\n        scores = []\n        for item in panel:\n            values = []\n            for document in item["candidates"]:\n                if time.time() >= config["eval_deadline"]:\n                    raise TimeoutError("Evaluation reserve reached; no partial metric is reported.")\n                values.extend(scorer.score([(item["question"], document["text"])]))\n            scores.append(values)\n        return ranking_metrics(panel, scores)\n\n    with scorer.model.disable_adapter():\n        before = measure()\n    after = measure()\n    atomic_json(\n        run / "reranking_validation.json",\n        {\n            "bm25": ranking_metrics(panel),\n            "base": before,\n            "fine_tuned": after,\n            "queries": len(panel),\n            "scope": "fixed validation subset with frozen BM25 candidates",\n            "gold_injection": False,\n            "test_set_used": False,\n            "quantization": "same NF4 base for both evaluations",\n        },\n    )\n'
QWEN_WORKER_SOURCE = QWEN_LIBRARY_SOURCE + '\nif __name__ == "__main__":\n    import sys\n    config = json.loads(Path(sys.argv[2]).read_text(encoding="utf-8"))\n    action = {"prepare": prepare_reranker_worker, "train": train_reranker_worker,\n              "evaluate": evaluate_reranker_worker}\n    action[sys.argv[1]](config)\n'
QWEN_RERANKER_SCORER_SOURCE = '"""Qwen3 yes/no reranking with the official prompt and a pinned optional LoRA adapter."""\n\nimport json\nfrom pathlib import Path\n\nSYSTEM = (\n    "Judge whether the Document meets the requirements "\n    "based on the Query and the Instruct provided. "\n    \'Note that the answer can only be "yes" or "no".\'\n)\nPREFIX = f"<|im_start|>system\\n{SYSTEM}<|im_end|>\\n<|im_start|>user\\n"\nSUFFIX = "<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n"\nDEFAULT_INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"\n\n\ndef pair_token_ids(tokenizer, query, document, instruction, max_length, *, truncate=False):\n    prefix = tokenizer.encode(PREFIX, add_special_tokens=False)\n    suffix = tokenizer.encode(SUFFIX, add_special_tokens=False)\n    body = tokenizer.encode(\n        f"<Instruct>: {instruction}\\n<Query>: {query}\\n<Document>: {document}",\n        add_special_tokens=False,\n    )\n    room = max_length - len(prefix) - len(suffix)\n    if room < 1:\n        raise ValueError("MAX_LENGTH is smaller than the reranker prompt overhead.")\n    if len(body) > room and not truncate:\n        raise ValueError("Query/document exceeds MAX_LENGTH; gold truncation is forbidden.")\n    return prefix + body[:room] + suffix\n\n\ndef read_reranker_manifest(path):\n    root = Path(path)\n    value = json.loads((root / "research_model.json").read_text(encoding="utf-8"))\n    if (\n        value.get("format") != "swift_qwen3_reranker_adapter_v1"\n        or value.get("task_type") != "generative_reranker"\n        or value.get("model_family") != "qwen3_reranker"\n        or value.get("base_model") != "Qwen/Qwen3-Reranker-4B"\n        or value.get("fine_tuned") is not True\n        or value.get("optimizer_steps", 0) < 1\n        or len(value.get("base_revision", "")) != 40\n        or value.get("positive_token") != "yes"\n        or value.get("negative_token") != "no"\n        or not (root / "adapter_config.json").is_file()\n        or not (root / "adapter_model.safetensors").is_file()\n    ):\n        raise ValueError("Invalid trained Qwen3 reranker artifact; base fallback is forbidden.")\n    return value\n\n\nclass QwenRerankerScorer:\n    def __init__(\n        self,\n        base_path,\n        adapter_path=None,\n        *,\n        instruction=DEFAULT_INSTRUCTION,\n        max_length=512,\n        device="cuda:0",\n        quantized=True,\n    ):\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n\n        self.instruction, self.max_length = instruction, max_length\n        self.tokenizer = AutoTokenizer.from_pretrained(\n            str(base_path), padding_side="left", trust_remote_code=False, local_files_only=True\n        )\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        self.yes = self.tokenizer.convert_tokens_to_ids("yes")\n        self.no = self.tokenizer.convert_tokens_to_ids("no")\n        if (\n            self.yes == self.no\n            or self.yes is None\n            or self.no is None\n            or self.tokenizer.encode("yes", add_special_tokens=False) != [self.yes]\n            or self.tokenizer.encode("no", add_special_tokens=False) != [self.no]\n        ):\n            raise ValueError("The tokenizer must represent yes and no as distinct single tokens.")\n        options = {\n            "trust_remote_code": False,\n            "local_files_only": True,\n            "attn_implementation": "sdpa",\n        }\n        if quantized:\n            if not device.startswith("cuda"):\n                raise ValueError("This NF4 recipe requires a CUDA GPU.")\n            options.update(\n                {\n                    "torch_dtype": torch.float16,\n                    "device_map": {"": device},\n                    "quantization_config": BitsAndBytesConfig(\n                        load_in_4bit=True,\n                        bnb_4bit_quant_type="nf4",\n                        bnb_4bit_use_double_quant=True,\n                        bnb_4bit_compute_dtype=torch.float16,\n                    ),\n                }\n            )\n        else:\n            options["torch_dtype"] = torch.float32 if device == "cpu" else torch.float16\n        model = AutoModelForCausalLM.from_pretrained(str(base_path), **options)\n        if not quantized:\n            model.to(device)\n        if adapter_path:\n            from peft import PeftModel\n\n            model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)\n        self.model = model.eval()\n\n    def score(self, pairs, batch_size=1, *, truncate=False):\n        import torch\n\n        if batch_size < 1:\n            raise ValueError("batch_size must be positive")\n        scores = []\n        with torch.inference_mode():\n            for start in range(0, len(pairs), batch_size):\n                ids = [\n                    pair_token_ids(\n                        self.tokenizer, q, d, self.instruction, self.max_length, truncate=truncate\n                    )\n                    for q, d in pairs[start : start + batch_size]\n                ]\n                batch = self.tokenizer.pad({"input_ids": ids}, padding=True, return_tensors="pt")\n                batch = batch.to(self.model.device)\n                # Left padding leaves the final position valid for every pair.\n                logits = (\n                    self.model(**batch, logits_to_keep=1, use_cache=False).logits[:, -1].float()\n                )\n                margin = logits[:, self.yes] - logits[:, self.no]\n                if not torch.isfinite(margin).all():\n                    raise RuntimeError("Non-finite reranker score.")\n                scores.extend(margin.cpu().tolist())\n        return scores\n\n\ndef load_reranker_adapter(path, device="cuda:0"):\n    from huggingface_hub import snapshot_download\n\n    manifest = read_reranker_manifest(path)\n    base = snapshot_download(manifest["base_model"], revision=manifest["base_revision"])\n    return QwenRerankerScorer(\n        base,\n        path,\n        instruction=manifest["instruction"],\n        max_length=manifest["max_length"],\n        device=device,\n    )\n'
exec(QWEN_LIBRARY_SOURCE, globals())

In [ ]:
# A private environment avoids changing Kaggle's already-loaded native libraries.
# Keep Kaggle's CUDA-enabled PyTorch via system-site-packages and an exact constraint.
SESSION_STARTED = session_start(
    "/kaggle/temp/qwen-session-clock.json", SESSION_ALREADY_ELAPSED_HOURS)
TRAIN_DEADLINE = SESSION_STARTED + 10 * 3600
TRAIN_HARD_DEADLINE = SESSION_STARTED + 10.5 * 3600
EVAL_DEADLINE = SESSION_STARTED + 11 * 3600
RUN = Path("/kaggle/working") / RUN_NAME
assert not RUN.exists(), "Choose a new RUN_NAME; never overwrite a previous run."
RUN.mkdir(parents=True)
ENV = Path("/kaggle/temp/qwen-training-env")
venv.EnvBuilder(with_pip=False, system_site_packages=True).create(ENV)
PYTHON = ENV / "bin/python"
constraints = ENV / "torch-constraints.txt"
torch_version = importlib.metadata.version("torch")
constraints.write_text("torch==" + torch_version + "\n", encoding="utf-8")
install_qwen_environment(PYTHON, QWEN_REQUIREMENTS, constraints, TRAIN_DEADLINE)
WORKER = Path("/kaggle/temp/qwen_reranker_worker.py")
WORKER.write_text(QWEN_WORKER_SOURCE, encoding="utf-8")
ADAPTER_MODULE = Path("/kaggle/temp/qwen_reranker_scorer.py")
ADAPTER_MODULE.write_text(QWEN_RERANKER_SCORER_SOURCE, encoding="utf-8")
preflight = r"""
import importlib.metadata as m, torch, numpy, pandas, datasets, bitsandbytes
from swift.llm import TrainArguments, sft_main
from swift.plugin import extra_callbacks
from transformers import Qwen3Model
from peft import PeftModel
from packaging.version import Version
assert torch.cuda.is_available(), "Enable Kaggle T4 GPU before Save Version"
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), (
    "Use a Kaggle image with PyTorch >=2.6 for safe optimizer-state resume")
assert torch.cuda.get_device_properties(0).total_memory >= 14 * 2**30
print("GPU:", torch.cuda.get_device_name(0))
print({p:m.version(p) for p in ["torch","ms-swift","transformers","peft","bitsandbytes"]})
"""
subprocess.run([str(PYTHON), "-c", preflight], check=True, timeout=180)
print("Session elapsed hours:", round((time.time() - SESSION_STARTED) / 3600, 2))

## Dữ liệu, prompt và kiểm tra chất lượng

QASPER giữ split theo paper; positive train không trùng held-out text. Mỗi group
có query role user, tài liệu positive/negative role assistant; system chứa custom
instruction, được SWIFT đưa vào `<Instruct>`. Negative BM25 trong cùng paper,
loại mọi gold đã biết và held-out duplicate. Gold có thể chưa đầy đủ, nên negative
vẫn là silver. Corpus riêng theo định dạng prepared-data của project.

Giới hạn 512 token tính **toàn bộ prompt ghép**, không phải 512 token riêng document.
Loại group quá dài, không cắt mất gold. Trước train, kiểm tra token IDs của SWIFT
khớp hoàn toàn helper inference cho cả positive và negative, labels đúng `[1, 0]`.

Validation loss chọn checkpoint; sau train so base và adapter trên cùng panel
16 query / tối đa 20 BM25 candidates mỗi query. Không chèn gold vào candidates,
báo candidate recall, nDCG@10/MRR@10/Recall@10. Đây là kiểm tra nhỏ trên validation,
không dùng test chọn model và không suy ra chất lượng tiếng Việt từ QASPER tiếng Anh.

In [ ]:
config = {
    "recipe": RECIPE, "run": str(RUN), "base_model": BASE_MODEL,
    "max_length": MAX_LENGTH, "learning_rate": LEARNING_RATE,
    "max_steps": MAX_STEPS, "pair_limit": PAIR_LIMIT, "instruction": INSTRUCTION,
    "prepared_data": PREPARED_DATA, "resume": RESUME_CHECKPOINT,
    "output_dir": str(RUN / "checkpoints"),
    "train_deadline": TRAIN_DEADLINE, "eval_deadline": EVAL_DEADLINE,
}
assert BASE_MODEL == "Qwen/Qwen3-Reranker-4B"
assert 128 <= MAX_LENGTH <= 512 and MAX_STEPS > 0 and PAIR_LIMIT >= 16
assert SESSION_ALREADY_ELAPSED_HOURS >= 0 and INSTRUCTION.strip()
atomic_json(RUN / "config.json", config)
status = run_bounded([str(PYTHON), str(WORKER), "prepare", str(RUN / "config.json")],
    RUN / "prepare.log", min(TRAIN_DEADLINE - 1800, time.time() + 3600))
if status["returncode"] != 0:
    print((RUN / "prepare.log").read_text(encoding="utf-8")[-8000:])
    raise RuntimeError("Preparation failed/timed out. Inspect prepare.log before training.")
config = json.loads((RUN / "config.json").read_text(encoding="utf-8"))
stats = json.loads((RUN / "data_statistics.json").read_text(encoding="utf-8"))
print({k:v for k,v in stats.items() if k != "provenance"})
print("Prompt check:", json.loads((RUN / "prompt_check.json").read_text(encoding="utf-8")))
print("Pinned base revision:", config["base_revision"])
print("Train remaining hours:", round((TRAIN_DEADLINE - time.time()) / 3600, 2))

In [ ]:
print("Training reranker; progress and logs are written continuously under", RUN)
train_status = run_bounded([str(PYTHON), str(WORKER), "train", str(RUN / "config.json")],
    RUN / "train.log", TRAIN_HARD_DEADLINE)
atomic_json(RUN / "process_status.json", train_status)
if train_status["returncode"] != 0:
    print((RUN / "train.log").read_text(encoding="utf-8")[-8000:])
    print("Preserving only complete, verified checkpoints if available.")
MODEL_OUT = export_reranker_adapter(config, train_status)
(MODEL_OUT / "qwen_reranker_scorer.py").write_text(QWEN_RERANKER_SCORER_SOURCE, encoding="utf-8")
shutil.make_archive(str(RUN / "qwen3-reranker-adapter"), "zip", MODEL_OUT)
print("Trained adapter:", MODEL_OUT)
print(json.loads((MODEL_OUT / "research_model.json").read_text(encoding="utf-8")))

In [ ]:
if time.time() < EVAL_DEADLINE - 300:
    evaluation_status = run_bounded(
        [str(PYTHON), str(WORKER), "evaluate", str(RUN / "config.json")],
        RUN / "evaluate.log", EVAL_DEADLINE)
    atomic_json(RUN / "evaluation_status.json", evaluation_status)
    result_path = RUN / "reranking_validation.json"
    if result_path.is_file():
        print(json.loads(result_path.read_text(encoding="utf-8")))
    else:
        print("Evaluation incomplete/skipped; no partial metric is reported. See evaluate.log.")
else:
    atomic_json(RUN / "evaluation_status.json", {"skipped": "export reserve"})
    print("Evaluation skipped to preserve output time.")

In [ ]:
from IPython.display import display, FileLink
for file in [RUN / "qwen3-reranker-adapter.zip", RUN / "config.json"]:
    display(FileLink(str(file)))
atomic_json(RUN / "session_report.json", {
    "session_limit_hours": 12, "train_stop_hours": 10, "train_hard_stop_hours": 10.5,
    "eval_stop_hours": 11, "elapsed_hours": (time.time() - SESSION_STARTED) / 3600,
    "optimizer_steps": read_reranker_manifest(MODEL_OUT)["optimizer_steps"],
    "resume_checkpoint": str(RUN / "resume"),
})
print("Download the adapter ZIP; retain the whole run folder for resume.")
print("Next session: add previous Output as Input, set RESUME_CHECKPOINT to its resume folder.")
print("Keep recipe/data unchanged; increase MAX_STEPS above the old global_step.")
print("Inference helper is included in the ZIP. Scores measure relevance, not factuality.")

## Dùng adapter sau khi train

Giải nén ZIP vào GPU host và dùng helper đi kèm, trong môi trường có các thư viện
đã pin ở trên. Base model đúng commit SHA sẽ được tải tự động; adapter nạp lỗi sẽ
báo lỗi, không âm thầm chuyển sang base model.

```python
import sys
sys.path.insert(0, "/path/to/extracted/qwen3-reranker-adapter")
from qwen_reranker_scorer import load_reranker_adapter
reranker = load_reranker_adapter("/path/to/extracted/qwen3-reranker-adapter", device="cuda:0")
scores = reranker.score([("research question", "candidate passage")], batch_size=1)
```

Đây là artifact **reranker**, có manifest/task riêng. Endpoint embedding trong
notebook 05 nhận artifact embedding; không dán adapter reranker vào đó.
Notebook này chưa triển khai API reranker/Cloudflare. Toàn bộ phiên Save & Run All
kết thúc sau khi lưu kết quả train và evaluation.